# Class Imbalance Techniques

This notebook loads the processed dataset, checks class imbalance, and applies common imbalance handling techniques before model training.

In [2]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

from imblearn.combine import SMOTEENN, SMOTETomek
from imblearn.over_sampling import ADASYN, BorderlineSMOTE, RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler, TomekLinks

In [3]:
def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()

    for path in [current, *current.parents]:
        # Use the data folder to identify the main project directory.
        if (path / 'data').exists():
            return path

    raise FileNotFoundError('Project root not found.')


# Define the paths used in this notebook.
BASE_DIR = find_project_root(Path.cwd())
DATA_PATH = BASE_DIR / 'data' / 'processed' / 'heart_disease_processed.csv'

BASE_DIR, DATA_PATH

(WindowsPath('D:/PROJECT OF DATA SCIENCE/Heart Disease Health Indicators'),
 WindowsPath('D:/PROJECT OF DATA SCIENCE/Heart Disease Health Indicators/data/processed/heart_disease_processed.csv'))

In [4]:
# Load the processed dataset and keep a working copy for experimentation.
data = pd.read_csv(DATA_PATH)
df = data.copy()
df.head()

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0,1,1,1,40.0,1,0,0.0,0,0,...,1,0,5.0,18.0,15.0,1,0,9.0,4.0,3.0
1,0,0,0,0,25.0,1,0,0.0,1,0,...,0,1,3.0,0.0,0.0,0,0,7.0,6.0,1.0
2,0,1,1,1,28.0,0,0,0.0,0,1,...,1,1,5.0,30.0,30.0,1,0,9.0,4.0,8.0
3,0,1,0,1,27.0,0,0,0.0,1,1,...,1,0,2.0,0.0,0.0,0,0,11.0,3.0,6.0
4,0,1,1,1,24.0,0,0,0.0,1,1,...,1,0,2.0,3.0,0.0,0,0,11.0,5.0,4.0


In [5]:
# Separate features and target before applying any imbalance technique.
target_column = 'HeartDiseaseorAttack'

X = df.drop(columns=[target_column])
y = df[target_column]

print(f'Feature shape: {X.shape}')
print(f'Target shape: {y.shape}')

Feature shape: (229781, 21)
Target shape: (229781,)


In [6]:
# Check the original class distribution so we know how imbalanced the data is.
original_distribution = y.value_counts().sort_index()
original_percentage = y.value_counts(normalize=True).sort_index() * 100

print('Original class distribution:')
print(original_distribution)
print('\nOriginal class percentage:')
print(original_percentage.round(2))

Original class distribution:
HeartDiseaseorAttack
0    206064
1     23717
Name: count, dtype: int64

Original class percentage:
HeartDiseaseorAttack
0    89.68
1    10.32
Name: proportion, dtype: float64


In [7]:
# Split the dataset before resampling to avoid data leakage from the test set.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')

X_train shape: (183824, 21)
X_test shape: (45957, 21)


In [8]:
def show_distribution(name, target):
    # This helper prints the class counts after each resampling method.
    counts = pd.Series(target).value_counts().sort_index()
    percentages = pd.Series(target).value_counts(normalize=True).sort_index() * 100

    print(f'\n{name}')
    print('Counts:')
    print(counts)
    print('Percentages:')
    print(percentages.round(2))

## 1. Cost-Sensitive Learning

This does not change the data itself. Instead, the model is trained later with `class_weight='balanced'` so the minority class gets more importance.

In [10]:
# Keep a note that class_weight is a data-free imbalance strategy.
class_weight_strategy = {'method': 'class_weight', 'value': 'balanced'}
class_weight_strategy

{'method': 'class_weight', 'value': 'balanced'}

## 2. Random Oversampling

This method duplicates minority-class samples until the classes become balanced.

In [11]:
random_oversampler = RandomOverSampler(random_state=42)
X_ros, y_ros = random_oversampler.fit_resample(X_train, y_train)
show_distribution('Random Oversampling', y_ros)


Random Oversampling
Counts:
HeartDiseaseorAttack
0    164850
1    164850
Name: count, dtype: int64
Percentages:
HeartDiseaseorAttack
0    50.0
1    50.0
Name: proportion, dtype: float64


## 3. Random Undersampling

This method removes majority-class samples until the classes become balanced.

In [12]:
random_undersampler = RandomUnderSampler(random_state=42)
X_rus, y_rus = random_undersampler.fit_resample(X_train, y_train)
show_distribution('Random Undersampling', y_rus)


Random Undersampling
Counts:
HeartDiseaseorAttack
0    18974
1    18974
Name: count, dtype: int64
Percentages:
HeartDiseaseorAttack
0    50.0
1    50.0
Name: proportion, dtype: float64


## 4. SMOTE

SMOTE creates synthetic minority-class samples instead of simply duplicating them.

In [13]:
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train, y_train)
show_distribution('SMOTE', y_smote)


SMOTE
Counts:
HeartDiseaseorAttack
0    164850
1    164850
Name: count, dtype: int64
Percentages:
HeartDiseaseorAttack
0    50.0
1    50.0
Name: proportion, dtype: float64


## 5. Borderline-SMOTE

This variation of SMOTE focuses on minority-class samples that are near the class boundary.

In [14]:
borderline_smote = BorderlineSMOTE(random_state=42)
X_bsmote, y_bsmote = borderline_smote.fit_resample(X_train, y_train)
show_distribution('Borderline-SMOTE', y_bsmote)


Borderline-SMOTE
Counts:
HeartDiseaseorAttack
0    164850
1    164850
Name: count, dtype: int64
Percentages:
HeartDiseaseorAttack
0    50.0
1    50.0
Name: proportion, dtype: float64


## 6. ADASYN

ADASYN creates more synthetic samples for harder minority cases and fewer for easier ones.

In [15]:
adasyn = ADASYN(random_state=42)
X_adasyn, y_adasyn = adasyn.fit_resample(X_train, y_train)
show_distribution('ADASYN', y_adasyn)


ADASYN
Counts:
HeartDiseaseorAttack
0    164850
1    166344
Name: count, dtype: int64
Percentages:
HeartDiseaseorAttack
0    49.77
1    50.23
Name: proportion, dtype: float64


## 7. Tomek Links

Tomek Links removes overlapping majority samples near the decision boundary.

In [16]:
tomek_links = TomekLinks()
X_tomek, y_tomek = tomek_links.fit_resample(X_train, y_train)
show_distribution('Tomek Links', y_tomek)


Tomek Links
Counts:
HeartDiseaseorAttack
0    159818
1     18974
Name: count, dtype: int64
Percentages:
HeartDiseaseorAttack
0    89.39
1    10.61
Name: proportion, dtype: float64


## 8. SMOTE + Tomek Links

This combined method first generates synthetic minority samples and then removes noisy boundary points.

In [ ]:
smote_tomek = SMOTETomek(random_state=42)
X_smote_tomek, y_smote_tomek = smote_tomek.fit_resample(X_train, y_train)
show_distribution('SMOTE + Tomek Links', y_smote_tomek)

## 9. SMOTE + ENN

This combined method first oversamples the minority class and then removes ambiguous samples using Edited Nearest Neighbours.

In [ ]:
smote_enn = SMOTEENN(random_state=42)
X_smote_enn, y_smote_enn = smote_enn.fit_resample(X_train, y_train)
show_distribution('SMOTE + ENN', y_smote_enn)

In [ ]:
# Collect all resampled datasets in one place so they are easy to reuse in model notebooks.
resampled_datasets = {
    'original_train': (X_train, y_train),
    'random_oversampling': (X_ros, y_ros),
    'random_undersampling': (X_rus, y_rus),
    'smote': (X_smote, y_smote),
    'borderline_smote': (X_bsmote, y_bsmote),
    'adasyn': (X_adasyn, y_adasyn),
    'tomek_links': (X_tomek, y_tomek),
    'smote_tomek': (X_smote_tomek, y_smote_tomek),
    'smote_enn': (X_smote_enn, y_smote_enn)
}

list(resampled_datasets.keys())

In [ ]:
# Use this summary table to compare how each technique changed the class balance.
summary = []

for name, (_, target_values) in resampled_datasets.items():
    counts = pd.Series(target_values).value_counts().sort_index()
    summary.append({
        'method': name,
        'class_0_count': counts.get(0, 0),
        'class_1_count': counts.get(1, 0)
    })

summary_df = pd.DataFrame(summary)
summary_df

## Which Technique Should Be Used?

- Start with `class_weight='balanced'` for Logistic Regression.
- Try `SMOTE`, `ADASYN`, and `SMOTEENN` for model comparison.
- Use resampling only on the training data, never on the test data.